## Lecture 9: Python Concurrency
### April 1, 2026

Partly based on [https://nyu-cds.github.io/python-concurrency/](https://nyu-cds.github.io/python-concurrency/)


## Improving performance by using concurrency

Concurrency vs parallelism:

    Concurrency is about dealing with lots of things at once. 
    
    Parallelism is about doing lots of things at once.
    
- Concurrency means executing multiple tasks at the same time but not necessarily simultaneously. <br><br>

- Parallelism means that an application splits its tasks up into smaller subtasks which can be processed in parallel, for instance on multiple CPUs at the exact same time. <br><br>
    
[source](https://medium.com/@itIsMadhavan/concurrency-vs-parallelism-a-brief-review-b337c8dac350) <br><br>


We will illustrate some benefits of concurrency with a program downloading images from the `imgur.com` website.<br><br>

In this Python concurrency tutorial, we will write a small Python script to download images from Imgur. Imgur is a popular image sharing site.

We will start with a version that downloads images sequentially, or one at a time, and then explore techniques for improving the performance of the program by introducing different forms of concurrency. <br><br>


For this you will need to:<br><br>

- create an account in [imgur.com](https://imgur.com/)
  - email:
  - Description:
  
This tutorial demonstrates how to create an OAuth application for use with the imgur API. This tutorial covers both: <br><br>
- (i) creating your application; as well as <br><br>
- (ii) retrieving your OAuth 2.0 client ID and client secret.<br><br>

Steps to follow:<br><br>
 
 https://tome.oauth.io/create-imgur-application.Imgur <br><br>
 
1-Sign in to imgur - https://imgur.com/ <br><br>

2-Navigate to register an OAuth application - https://imgur.com/account/settings/apps <br><br>



---
The functions below fetchs a list of images and download them __imgur__ repository: <br><br>
[https://imgur.com/](https://imgur.com/)

- We will start with a version that downloads images sequentially, or one at a time <br><br>

- Then improve the performance by introducing multiprocessing and threading <br><br>

---
We will split the functionality into three separate functions, see the file `download.py` <br><br>
- `get_links(...)`: is used to obtain a list of available images. <br><br>
- `download_link(...)`: downloads the image given by the URL link into the download directory. <br><br>
- `setup_download_dir()`: function creates a download destination directory if it doesn’t already exist. <br><br>

Next, we will use these functions to download the images, one by one. <br><br>

This will contain the main function of our first, naive version of the Imgur image downloader.

The module will use the Imgur client ID that you previously obtained.

By invoking the `setup_download_dir()` to create the download destination directory.

Then it will fetch a list of images using the `get_links(...)` function, filter out all GIF and album URLs, and finally use `download_link(...)` to download and save each of those images to the disk.

In [1]:
from time import time

# 56a8f8b3a123bd9 'replace with your client ID'
CLIENT_ID = '56a8f8b3a123bd9'
from download import setup_download_dir, get_links, download_link

ts = time()
download_dir = setup_download_dir()

links = [l for l in get_links(CLIENT_ID)]

for i, link in enumerate(links):
    print("%2d %s" % (i, link))
    download_link(download_dir, link)

print('Took {}s'.format(time() - ts))

 0 http://i.imgur.com/W51YgmWh.gif
 1 https://i.imgur.com/W5FSGMV.gif
 2 https://i.imgur.com/W5Bw0uu.gif
 3 https://i.imgur.com/W5Jv5.jpg
 4 http://i.imgur.com/W5J2zYth.gif
 5 http://i.imgur.com/W53xXpoh.gif
 6 http://i.imgur.com/W5r8loWh.gif
 7 https://i.imgur.com/W5VLGjV.png
 8 https://i.imgur.com/W5JX1sx.jpg
 9 http://i.imgur.com/W5E0SdTh.gif
10 https://i.imgur.com/W5rh4.jpg
11 https://i.imgur.com/W5Rv2SR.jpg
12 https://i.imgur.com/W5Ny2Cr.jpg
13 https://i.imgur.com/W5jraMP.jpg
14 https://i.imgur.com/W5glh.jpg
Took 0.8856089115142822s


In [2]:
!ls images

W51YgmWh.gif W5E0SdTh.gif W5JX1sx.jpg  W5Rv2SR.jpg  W5jraMP.jpg
W53xXpoh.gif W5FSGMV.gif  W5Jv5.jpg    W5VLGjV.png  W5r8loWh.gif
W5Bw0uu.gif  W5J2zYth.gif W5Ny2Cr.jpg  W5glh.jpg    W5rh4.jpg


---


The good news is that by introducing concurrency or parallelism, we can speed this up dramatically.

---

- To improve the performance of the image downloader we can run **multiple copies** of the program at the same time.  <br><br>


- However, we would need to know what images are available so that we could ensure that one process didn’t download an image that had already been downloaded by a different process.   <br><br>


- Fortunately the `multiprocessing` module is available for this purpose. <br><br>

---

### Pool

- To use multiple processes we need a `multiprocessing`'s `Pool`.  <br><br>


- The `Pool` class provides a `map` method that runs a function as a separate process, passing arguments from a supplied iterable.  <br><br>


- The iterable is divided into a number of chunks, so that each process gets roughly the same number of elements. <br><br> 


- We will pass the list of URLs to the pool, which starts 8 new processes and use each one to download the images in parallel. <br><br>

In [3]:
from multiprocessing import cpu_count
print("number of CPU cores:", cpu_count())

number of CPU cores: 10


In [4]:
from functools import partial
from multiprocessing.pool import Pool

def multi_processes_download():
    ts = time()
    download_dir = setup_download_dir()
    links = [l for l in get_links(CLIENT_ID)]

    # functools.partial makes a new version of a function 
    # with one or more fixed arguments
    download = partial(download_link, download_dir)
   
    with Pool(8) as p:
        p.map(download, links)
        
    print('Took {}s'.format(time() - ts))

multi_processes_download()

Took 0.6294300556182861s


---

This is true parallelism, but it comes with a cost.

Although easy to implement, the parallelism bears some drawbacks: <br><br>

- each process contains **a copy of the entire memory** <br><br>
- it does not handle processes that depend on each other <br><br>

Those issues can be tackled by shared memory and message passing mechanisms, which we will learn from later lessons.<br><br>

## Using Threads

Threading is a well known approach to attaining concurrency: <br><br> 
- typically threads are lighter weight than processes <br><br>
- **lower memory requirements**, as **they share the same memory space** <br><br>

A basic way to use threads is through `ThreadPoolExecutor` in `concurrent.futures`, which provides a similar interface to `multiprocessing.Pool`. <br><br>

For more refined behavior will rely on the `Thread` class, which provides a `run` method that should be overridden with a method that does the actual work of the thread. <br><br>

In this lesson, we will use Pyton threading to increase the performance of our image downloader. <br><br> 

To do this, we will create a pool of 8 threads, making a total of 9 threads including the main thread.  <br><br>

The optimal number of threads is usually chosen based on factors such as other applications and services running on the same machine. <br><br>

We will chose 8 worker threads because many personal computers have 8 CPU cores and one worker thread per core seems like a good number for how many threads to run at once. <br><br>

In [5]:
## Simple example with ThreadPoolExecutor

from functools import partial
from concurrent.futures import ThreadPoolExecutor

def multithreaded_download():
    ts = time()
    download_dir = setup_download_dir()
    links = [l for l in get_links(CLIENT_ID)]

    download = partial(download_link, download_dir)
   
    with ThreadPoolExecutor(max_workers=8) as ex:
        ex.map(download, links)
        
    print('Took {}s'.format(time() - ts))

multithreaded_download()

Took 0.6362631320953369s


### Thread Safety

- Each thread shares the same memory space.

- Variables in the program are shared by all the threads and should not be accessed the way you would normally access a variable.  <br><br>
- One thread may change the variable while another thread is reading it, or worse, two threads may try to update the variable at the same time.  <br><br>


- This is known as a **race condition**, it is one of the leading sources of errors in threaded programs and needs to be addressed properly. <br><br>



- A way to deal with thread safety is using the __Queue Class__

In [6]:
# Understanding Queue 
from queue import Queue

def do_work(q):
    while not q.empty():
        item = q.get()
        print(str(item)) 
        q.task_done()  # this is important when combining Queue with Threads

q = Queue() # FIFO queue

for i in range(20):
    q.put(i)

do_work(q)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19


A simpler example before going back to the image downloader code

In [7]:
# in this example each thread prints an element of the queue

from time import sleep
from queue import Queue
from threading import Thread
import logging  

# set up a logger
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logging.basicConfig(format='(%(threadName)-9s) %(message)s', level=logging.DEBUG)

def do_work(q):
    while True:
        item = q.get()
        logger.debug("e" + str(item) + ' ')
        print(str(item) + ' ')
        q.task_done()   # this has to called for each and every item that has been inserted to the queue
        sleep(2)
    
q = Queue()
num_threads = 10

for i in range(num_threads):
    worker = Thread(target=do_work, args=(q,), name='thread_' + str(i))
    worker.setDaemon(True) # this stop the threads when the program quits  
    worker.start()         # start the threads

# now we have started 10 threads:

for i in range(50):
    q.put(i)

q.join() # wait until all threads have finished (waits until all 50 q.task_done() have been called)

/var/folders/vj/mhdkxj391ljd_nk06v7m8hm40000gn/T/ipykernel_56336/2911971276.py:26: DeprecationWarning: setDaemon() is deprecated, set the daemon attribute instead
  worker.setDaemon(True) # this stop the threads when the program quits
(thread_0 ) e0 
(thread_1 ) e1 
(thread_2 ) e2 
(thread_3 ) e3 
(thread_4 ) e4 
(thread_5 ) e5 
(thread_6 ) e6 
(thread_7 ) e7 
(thread_8 ) e8 
(thread_9 ) e9 


0 
1 
2 
3 
4 
5 
6 
7 
8 
9 


(thread_0 ) e10 
(thread_2 ) e11 
(thread_3 ) e12 
(thread_6 ) e13 
(thread_7 ) e14 
(thread_9 ) e15 
(thread_5 ) e16 
(thread_8 ) e17 
(thread_1 ) e18 
(thread_4 ) e19 


10 
11 
12 
13 
14 
15 
16 
17 
18 
19 


(thread_2 ) e20 
(thread_0 ) e21 
(thread_6 ) e22 
(thread_3 ) e23 
(thread_9 ) e24 
(thread_5 ) e25 
(thread_8 ) e26 
(thread_7 ) e27 
(thread_1 ) e28 
(thread_4 ) e29 


20 
21 
22 
23 
24 
25 
26 
27 
28 
29 


(thread_2 ) e30 
(thread_6 ) e31 
(thread_9 ) e32 
(thread_0 ) e33 
(thread_3 ) e34 
(thread_5 ) e35 
(thread_8 ) e36 
(thread_7 ) e37 
(thread_1 ) e38 
(thread_4 ) e39 


30 
31 
32 
33 
34 
35 
36 
37 
38 
39 


(thread_2 ) e40 
(thread_6 ) e41 
(thread_9 ) e42 
(thread_0 ) e43 
(thread_8 ) e44 
(thread_4 ) e45 
(thread_3 ) e46 
(thread_7 ) e47 
(thread_1 ) e48 
(thread_5 ) e49 


40 
41 
42 
43 
44 
45 
46 
47 
48 
49 


the image downloader code

In [8]:
from queue import Queue
from threading import Thread

class DownloadWorker(Thread):
    def __init__(self, queue):
        super(DownloadWorker, self).__init__()
        self.queue = queue
    
    def run(self):
        while True:
            # Get the work from the queue and expand the tuple
            (directory, link) = self.queue.get()
            # call the function donwload_link (from download.py)
            download_link(directory, link)
            self.queue.task_done()

            
def threaded_download():
    ts = time()
    download_dir = setup_download_dir()
    links = [l for l in get_links(CLIENT_ID)]
    
    # Create a queue to communicate with the worker threads
    queue = Queue()
    
    # Create 8 worker threads
    for _ in range(8):
        worker = DownloadWorker(queue)
        # Setting daemon to True will let the main thread exit 
        # even if the workers are blocking
        worker.daemon = True
        worker.start()

    
    # Put the tasks into the queue as a tuple
    for link in links:
        print('Queueing: {}'.format(link))
        queue.put((download_dir, link))
    
    # Causes the main thread to wait for the queue to finish processing all the tasks
    queue.join()
    
    print('Took {}s'.format(time() - ts))

threaded_download()

(MainThread) Starting new HTTPS connection (1): api.imgur.com:443
(MainThread) https://api.imgur.com:443 "GET /3/gallery/random/random/ HTTP/1.1" 200 23072
(Thread-8 ) Starting new HTTP connection (1): i.imgur.com:80
(Thread-10) Starting new HTTPS connection (1): i.imgur.com:443
(Thread-12) Starting new HTTP connection (1): i.imgur.com:80
(Thread-11) Starting new HTTPS connection (1): i.imgur.com:443
(Thread-7 ) Starting new HTTP connection (1): i.imgur.com:80
(Thread-9 ) Starting new HTTPS connection (1): i.imgur.com:443
(Thread-13) Starting new HTTPS connection (1): i.imgur.com:443
(Thread-14) Starting new HTTP connection (1): i.imgur.com:80
(Thread-8 ) http://i.imgur.com:80 "GET /W51YgmWh.gif HTTP/1.1" 429 0
(Thread-8 ) Starting new HTTPS connection (1): i.imgur.com:443
(Thread-14) http://i.imgur.com:80 "GET /W53xXpoh.gif HTTP/1.1" 429 0
(Thread-12) http://i.imgur.com:80 "GET /W5J2zYth.gif HTTP/1.1" 429 0
(Thread-7 ) http://i.imgur.com:80 "GET /W5r8loWh.gif HTTP/1.1" 429 0
(Thread-1

Queueing: http://i.imgur.com/W51YgmWh.gif
Queueing: https://i.imgur.com/W5FSGMV.gif
Queueing: https://i.imgur.com/W5Bw0uu.gif
Queueing: https://i.imgur.com/W5Jv5.jpg
Queueing: http://i.imgur.com/W5J2zYth.gif
Queueing: http://i.imgur.com/W53xXpoh.gif
Queueing: http://i.imgur.com/W5r8loWh.gif
Queueing: https://i.imgur.com/W5VLGjV.png
Queueing: https://i.imgur.com/W5JX1sx.jpg
Queueing: http://i.imgur.com/W5E0SdTh.gif
Queueing: https://i.imgur.com/W5rh4.jpg
Queueing: https://i.imgur.com/W5Rv2SR.jpg
Queueing: https://i.imgur.com/W5Ny2Cr.jpg
Queueing: https://i.imgur.com/W5jraMP.jpg
Queueing: https://i.imgur.com/W5glh.jpg


(Thread-10) https://i.imgur.com:443 "GET /W5Bw0uu.gif HTTP/1.1" 429 0
(Thread-10) Starting new HTTPS connection (1): i.imgur.com:443
(Thread-13) https://i.imgur.com:443 "GET /W5VLGjV.png HTTP/1.1" 429 0
(Thread-13) Starting new HTTPS connection (1): i.imgur.com:443
(Thread-11) https://i.imgur.com:443 "GET /W5Jv5.jpg HTTP/1.1" 429 0
(Thread-9 ) https://i.imgur.com:443 "GET /W5FSGMV.gif HTTP/1.1" 429 0
(Thread-8 ) https://i.imgur.com:443 "GET /W5JX1sx.jpg HTTP/1.1" 429 0
(Thread-12) https://i.imgur.com:443 "GET /W5rh4.jpg HTTP/1.1" 429 0
(Thread-7 ) https://i.imgur.com:443 "GET /W5Rv2SR.jpg HTTP/1.1" 429 0
(Thread-14) https://i.imgur.com:443 "GET /W5Ny2Cr.jpg HTTP/1.1" 429 0
(Thread-10) https://i.imgur.com:443 "GET /W5jraMP.jpg HTTP/1.1" 429 0
(Thread-13) https://i.imgur.com:443 "GET /W5glh.jpg HTTP/1.1" 429 0


Took 0.656973123550415s


So we have seen two versions of the threaded downloader. One with a `ThreadPoolExecutor` and one with a `Thread` class. Usually the former is preferred because it's simpler and likely to have fewer bugs since Python handles the low level details, such as thread management, shutdown, and error handling. The latter is useful and more flexible when you need custom behavior (long-running workers, etc.), fine control over the threads, and etc.

## The Global Interpreter Lock

In multithreading, a **lock** is a mechanism for preventing multiple threads from accessing the same shared variable simultaneously. 

This is to avoid a race condition, as mentioned previously. <br><br>
 
#### Not really parallel (not true parallelism)!

- Python has a **Global Interpreter Lock (GIL)**, (To prevent the execution of multiple threads at once in the Python interpreter) which allows only **one thread to be executed at a time** throughout this process. Therefore, **this code is concurrent but not parallel**.  <br><br>

- The reason it is still faster is because the image downloader is an **input/output (I/O) bound task**.  <br><br>
The majority of the time is spent waiting for the network. This is why threading can provide a large speed increase.  <br><br>

- **The processor can switch between the threads** whenever one of them is **ready** to do some work. <br><br>



- If the program was performing a task that was CPU bound, using the threading module in Python or any other interpreted language with a GIL could actually result in reduced performance. <br><br>

- For CPU bound tasks and truly parallel execution in Python, the multiprocessing module is a better option. <br><br>

- Some parallelism is still possible with threads if the executed functions rely on low-level code that realeases the GIL (e.g. many Numpy/Scipy functions). This includes custom Cython programs (see the `nogil` keyword [here] <br><br>(https://cython.readthedocs.io/en/latest/src/userguide/parallelism.html) and [here](https://cython.readthedocs.io/en/latest/src/userguide/numpy_tutorial.html)) <br><br>

- Other packages for parallelization: task/job queues (e.g. [python-rq](https://python-rq.org/)), [joblib](https://joblib.readthedocs.io/en/latest/parallel.html), [dask](https://dask.org/)


### Example: sum of array elements in parallel

In [9]:
n = int(1e8)

In [10]:
# Sequential version
from time import time

ts = time()
s = 0
for i in range(n):
    s = s + i
print(s, '-->', time()-ts,'s')   

4999999950000000 --> 4.609957933425903 s


In [11]:
# multiprocessing version
from time import time
from multiprocessing.pool import Pool
# we need to define these functions in a separate file to be able to use in the multiprocessing/multithreading version
from download import sum_multi_processes_1, sum_multi_processes_2


def sum_multi_processes_1_(chunk):
    y = 0
    for i in chunk:
        y = y + i
    return y


def sum_multi_processes_2_(start, end):
    y = 0
    for i in range(start, end):
        y = y + i
    return y

chunks1 = [list(range(i,i + 100)) for i in range(0, n, 100)]
print(len(chunks1), 'chunks')

ts = time()
with Pool(8) as p:
     results = p.map(sum_multi_processes_1, chunks1)

print(sum(results), '-->', time()-ts,'s')   

1000000 chunks
4999999950000000 --> 3.16029691696167 s


In [12]:
chunks2 = [(i,i + 100) for i in range(0, n, 100)]
print(len(chunks2), 'chunks')

ts = time()
with Pool(8) as p:
    results = p.starmap(sum_multi_processes_2, chunks2)

print(sum(results), '-->', time()-ts,'s')

1000000 chunks
4999999950000000 --> 1.190575122833252 s


We can easily notice that providing only the start and end index of each chunk as the function argument(s) is more efficient than providing the entire chunk as list.

In [13]:
# Thread version
from queue import Queue
from threading import Thread
from threading import Lock

x = 0
lock = Lock()
def sum_chunk(q):
    while True:
        global x
        start, end = q.get()
        for i in range(start, end):
            with lock:  # force synchronization
                x = x + i
        q.task_done()

n = int(1e8)
chunks = [(i, i + 100) for i in range(0, n, 100)]

ts = time()
q = Queue()
num_threads = 10

for i in range(num_threads):
    worker = Thread(target=sum_chunk, args=(q, ))
    worker.setDaemon(True) # this stop the threads when the program quits  
    worker.start()         # start the threads

for chunk in chunks:
    q.put(chunk)

q.join()
print(x, '-->', time() - ts, 's')    

/var/folders/vj/mhdkxj391ljd_nk06v7m8hm40000gn/T/ipykernel_56336/40114804.py:26: DeprecationWarning: setDaemon() is deprecated, set the daemon attribute instead
  worker.setDaemon(True) # this stop the threads when the program quits


4999999950000000 --> 23.66919493675232 s


Since summing massive consecutive integers is a CPU-bound task, we can clearly see that the threading version is much slower than the multiprocessing version.

### Example: Pi Simulation

In [14]:
import numpy as np
from download import monte_carlo_pi

def monte_carlo_pi_(n):
    s = 0
    for i in range(n):
        x = np.random.uniform(0, 1)
        y = np.random.uniform(0, 1)
        if (x**2 + y**2) < 1:
            s += 1
    return 4*s/n

In [15]:
%%time
result = [monte_carlo_pi(int(3e5)) for _ in range(10)]

CPU times: user 5.8 s, sys: 35.3 ms, total: 5.83 s
Wall time: 5.88 s


In [16]:
np.array(result)

array([3.14024   , 3.14310667, 3.14508   , 3.14178667, 3.14049333,
       3.14184   , 3.14242667, 3.13709333, 3.14382667, 3.14590667])

In [17]:
from multiprocessing.pool import Pool

In [18]:
%%time
with Pool(8) as pool:
    result = pool.map(monte_carlo_pi, [int(3e5) for _ in range(10)])

CPU times: user 7.44 ms, sys: 191 ms, total: 199 ms
Wall time: 1.9 s


In [19]:
np.array(result)

array([3.14046667, 3.14476   , 3.14209333, 3.14069333, 3.13841333,
       3.14185333, 3.14358667, 3.14432   , 3.14178667, 3.13677333])